# Aprendizado de Máquina — Lista prática 05

## Métodos Não Paramétricos: Aspectos Teóricos

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta é a única lista do curso sem conjunto de dados: tudo aqui é simulação, porque
o assunto da aula é uma afirmação sobre **geometria em dimensão alta**, e geometria
se mede sorteando pontos.

> **a maldição da dimensionalidade não é uma metáfora. Ela é uma conta sobre
> volume, e você vai reproduzi-la em quatro medições.**

As duas primeiras batem com a fórmula até a terceira casa. A terceira não bate —
e a parte interessante é entender por quê.

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots
from scipy.special import gamma

from sklearn.neighbors import NearestNeighbors, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — quase todo ponto está na casca

Comece pelo fato mais simples e mais brutal. Sorteie pontos no cubo $[0,1]^p$ e
conte quantos estão a menos de $\varepsilon$ de **alguma** face.

Um ponto está perto da borda se **pelo menos uma** das suas coordenadas for
menor que $\varepsilon$ ou maior que $1-\varepsilon$. A probabilidade de isso
**não** acontecer é $(1-2\varepsilon)^p$, uma coordenada de cada vez.

In [ ]:
rng = np.random.default_rng(2026)
eps = 0.05
dims = [1, 2, 5, 10, 20, 50]

print("   d    medido   previsto")
for d in dims:
    X = rng.uniform(0, 1, size=(20000, d))
    perto = np.mean(...)   # (a)
    previsto = ...                                    # (b)
    print(f"  {d:3d}   {perto:.4f}    {previsto:.4f}")

---
## Exercício 2 — o vizinho mais próximo fica longe

A nota estima o raio necessário para reunir $k$ vizinhos por
$\rho \approx (k/n)^{1/p}$, tratando o volume de uma bola de raio $\rho$ como
$\rho^p$. O volume verdadeiro tem uma constante:

$$V_p(\rho) = \frac{\pi^{p/2}}{\Gamma\!\left(\frac p2 + 1\right)}\,\rho^p .$$

Vamos medir a distância ao vizinho mais próximo e comparar com as **duas**
previsões.

In [ ]:
def volume_bola_unitaria(d):
    return ...                 # (a)


rng = np.random.default_rng(2026)
n = 1000

print("   d   medido   heuristica   com o volume da bola      V_d")
for d in dims:
    X = rng.uniform(0, 1, size=(n, d))
    # o vizinho mais proximo de cada ponto e o SEGUNDO da lista (o primeiro e ele mesmo)
    dist, _ = NearestNeighbors(n_neighbors=2).fit(X).kneighbors(X)
    medido = np.median(...)                             # (b)

    heuristica = (1 / n) ** (1 / d)
    Vd = volume_bola_unitaria(d)
    com_volume = ...                     # (c)

    print(f"  {d:3d}   {medido:.4f}    {heuristica:.4f}       {com_volume:.4f}"
          f"          {Vd:.3g}")

> **Sua vez.** Repita a medição com $n=10\,000$ em vez de $1\,000$. Em $p=1$ a
> distância cai por um fator de 10, como esperado. Em $p=50$, por quanto ela cai?

---
## Exercício 3 — a taxa $n^{-2/(2+p)}$, medida

Agora o teorema em si. Vamos medir o erro do KNN contra $n$, em três dimensões, e
estimar o expoente por regressão nos logaritmos: se $\mathrm{EQM}\approx C n^{a}$,
então $\log \mathrm{EQM} \approx \log C + a\log n$, e $a$ é a inclinação.

A função de regressão tem curvatura em **todas** as coordenadas e constante de
Lipschitz que não cresce com $p$:

$$r(x) = \frac{1}{\sqrt p}\sum_{j=1}^{p}\operatorname{sen}(2\pi x_j).$$

Como conhecemos $r$, medimos o erro contra ela e não contra $y$ — o $\sigma^2$ é
uma constante aditiva que esconderia o expoente.

In [ ]:
def r(X):
    return np.sin(2 * np.pi * X).sum(axis=1) / np.sqrt(X.shape[1])


SIGMA = 0.3
ns = np.array([250, 500, 1000, 2000, 4000])
ks = np.array([1, 2, 3, 5, 8, 13, 21, 34, 55, 89])

resultados = {}
for d in (1, 5, 10):
    rng = np.random.default_rng(2026)
    eqm = []
    for n in ns:
        acc = []
        for _ in range(4):
            X = rng.uniform(0, 1, size=(n, d))
            y = r(X) + rng.normal(0, SIGMA, size=n)
            X0 = rng.uniform(0, 1, size=(2000, d))
            r0 = ...                                          # (a) o alvo verdadeiro
            # o melhor k possivel: medimos a taxa do metodo, sem o erro de escolher k
            acc.append(min(
                np.mean((KNeighborsRegressor(n_neighbors=int(k)).fit(X, y).predict(X0) - r0) ** 2)
                for k in ks if k <= n))
        eqm.append(np.mean(acc))

    eqm = np.array(eqm)
    resultados[d] = eqm
    expoente = np.polyfit(..., ..., 1)[0]      # (b) e (c)
    print(f"d={d:2d}: " + "  ".join(f"{v:.5f}" for v in eqm)
          + f"   expoente {expoente:+.3f}   cota {-2 / (2 + d):+.3f}")

---
## Exercício 4 — a fuga: supor estrutura

A maldição vale para quem não supõe nada sobre $r$. A nossa $r$ é, por
construção, **aditiva**: uma soma de funções de uma variável cada.

Um método que sabe disso pode estimar cada parcela separadamente — e cada uma é
um problema unidimensional, imune à maldição. É o que faz um modelo aditivo com
*splines*: `SplineTransformer` gera as bases de cada coordenada isoladamente, e a
regressão linear combina todas.

Compare os dois em $p=10$.

In [ ]:
d, n = 10, 2000
rng = np.random.default_rng(2026)
erros_knn, erros_aditivo = [], []

for _ in range(6):
    X = rng.uniform(0, 1, size=(n, d))
    y = r(X) + rng.normal(0, SIGMA, size=n)
    X0 = rng.uniform(0, 1, size=(3000, d))
    r0 = r(X0)

    erros_knn.append(min(
        np.mean((KNeighborsRegressor(n_neighbors=int(k)).fit(X, y).predict(X0) - r0) ** 2)
        for k in ks if k <= n))

    aditivo = make_pipeline(
        SplineTransformer(n_knots=..., degree=3),                # (a) nos por coordenada
        LinearRegression(),
    ).fit(X, y)
    erros_aditivo.append(np.mean((... - r0) ** 2))   # (b)

print(f"KNN pleno (melhor k):        EQM {np.mean(erros_knn):.5f}")
print(f"aditivo (splines por coord): EQM {np.mean(erros_aditivo):.5f}")
print(f"razao: {...:.1f}x")   # (c)

> **Sua vez.** Troque a função de regressão por
> $r(x) = \operatorname{sen}(2\pi x_1 x_2)$, que **não** é aditiva, e repita a
> comparação em $p=10$. O modelo aditivo continua ganhando?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | com uma casca de 5%, em $p=50$ **99,5%** dos pontos estão nela — e a fórmula acerta em todas as dimensões |
| 2 | o volume da bola unitária cresce até $p=5$ e depois desaba: $10^{-13}$ em $p=50$ |
| 2 | de $p=20$ em diante, as duas fórmulas subestimam o raio — porque a bola já não cabe no cubo |
| 3 | o expoente medido cai como previsto ($-0{,}74$, $-0{,}39$, $-0{,}21$), mas é sempre **melhor** que a cota |
| 4 | supor aditividade divide o erro por **34** em $p=10$, sem um dado a mais |

**A seguir.** A Aula 06 traz uma família que faz uma suposição diferente — $r$
aproximadamente constante por blocos — e um truque para reduzir a variância de
estimadores instáveis sem tocar no viés deles.